# 05 - Validation & Monitoring
Targets: **Gini >= 0.50, AUC >= 0.75, KS >= 40%**; PSI < 0.10 stable / 0.10-0.25 review / >= 0.25 recalibrate.

In [1]:
import os, sys; os.chdir('..') if os.path.basename(os.getcwd())=='notebooks' else None; sys.path.insert(0,'.')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, joblib, matplotlib.pyplot as plt
%matplotlib inline
from src.metrics import gini, auc, ks, psi, csi, psi_flag
from sklearn.metrics import roc_auc_score
train, test = joblib.load('data/processed/split.joblib'); art = joblib.load('data/processed/scorecard.joblib'); sc = art['scorecard']
s_tr, s_te = sc.score(train), sc.score(test)
T = {'Gini': (.50, gini), 'AUC': (.75, auc), 'KS': (.40, ks)}
rows = [(m, thr, f(train.default, -s_tr), f(test.default, -s_te), 'PASS' if f(test.default, -s_te) >= thr else 'FAIL') for m, (thr, f) in T.items()]
display(pd.DataFrame(rows, columns=['metric', 'threshold', 'train', 'test', 'test status']).round(4))
rng = np.random.default_rng(1); y = test.default.values; sv = -s_te; g = [gini(y[i], sv[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(300))]
print('Test Gini 95%% bootstrap CI: [%.4f, %.4f]' % tuple(np.percentile(g, [2.5, 97.5])))

,metric,threshold,train,test,test status
0,Gini,0.50,0.5492,0.5229,PASS
1,AUC,0.75,0.7746,0.7615,PASS
2,KS,0.40,0.4151,0.3964,FAIL


Test Gini 95% bootstrap CI: [0.4939, 0.5463]


## Stability: PSI (train vs test) and CSI per variable

In [2]:
p = psi(s_tr, s_te); print('Score PSI train->test: %.4f  -> %s' % (p, psi_flag(p)))
cs = pd.DataFrame({'CSI': {v: csi(sc.b[v], train[v].values, test[v].values) for v in sc.points}}); display(cs.round(4))

Score PSI train->test: 0.0010  -> Stable


,CSI
LIMIT_BAL,0.0009
PAY_1,0.0003
PAY_3,0.0000
PAY_5,0.0000
PAY_6,0.0002
PAY_AMT1,0.0009
PAY_AMT2,0.0004
PAY_AMT3,0.0015
PAY_AMT4,0.0011
PAY_AMT6,0.0011


## Drift simulation (SYNTHETIC shift)
To show the monitor firing, the test population is resampled so already-delinquent customers (PAY_1 >= 1) are 3x as likely to appear, mimicking a deteriorating intake. This is a **simulated** scenario, not observed drift.

In [3]:
w = 1 + 2*(test.PAY_1 >= 1).values; idx = rng.choice(len(test), len(test), p=w / w.sum()); shift = test.iloc[idx]; s_sh = sc.score(shift)
psh = psi(s_tr, s_sh); print('Shifted-population score PSI: %.4f -> %s' % (psh, psi_flag(psh)))
print('Bad rate: test %.3f -> shifted %.3f | Gini under shift %.4f' % (test.default.mean(), shift.default.mean(), gini(shift.default, -s_sh)))
pp, oo = sc.pd_from_score(s_sh).mean(), shift.default.mean(); print('Mean predicted PD under shift %.3f vs observed %.3f -> %s' % (pp, oo, 'calibration holds (pure covariate shift: P(bad|X) unchanged) - PSI flags the population change, no recalibration needed' if abs(pp-oo) < 0.02 else 'miscalibrated - recalibrate'))
display(pd.DataFrame({'CSI under shift': {v: csi(sc.b[v], train[v].values, shift[v].values) for v in sc.points}}).round(3))

Shifted-population score PSI: 0.2065 -> Moderate drift - review
Bad rate: test 0.221 -> shifted 0.308 | Gini under shift 0.5990
Mean predicted PD under shift 0.314 vs observed 0.308 -> calibration holds (pure covariate shift: P(bad|X) unchanged) - PSI flags the population change, no recalibration needed


,CSI under shift
LIMIT_BAL,0.012
PAY_1,0.245
PAY_3,0.053
PAY_5,0.032
PAY_6,0.021
PAY_AMT1,0.057
PAY_AMT2,0.024
PAY_AMT3,0.019
PAY_AMT4,0.014
PAY_AMT6,0.007


## Decile table (test) and API check

In [4]:
t = test.assign(score=s_te); t['decile'] = pd.qcut(-t.score, 10, labels=range(1, 11)); dt = t.groupby('decile').agg(accounts=('default', 'size'), bad_rate=('default', 'mean'), min_score=('score', 'min'), max_score=('score', 'max')); dt['cum_bad_capture'] = (dt.bad_rate*dt.accounts).cumsum() / t.default.sum(); display(dt.round(3))

,accounts,bad_rate,min_score,max_score,cum_bad_capture
decile,,,,,
1,900,0.060,558.609,578.938,0.027
2,900,0.090,552.628,558.603,0.068
3,900,0.080,548.087,552.627,0.104
4,901,0.128,543.777,548.085,0.162
5,899,0.143,539.802,543.767,0.227
6,900,0.164,535.197,539.799,0.301
7,900,0.211,527.940,535.195,0.396
8,900,0.231,506.185,527.924,0.501
9,900,0.438,481.039,506.174,0.699


In [5]:
from fastapi.testclient import TestClient
from src.app import app
from src.features import ST, BILLS, PAYS
c = TestClient(app); print(c.get('/health').json())
row = test.iloc[0]; payload = {k: float(row[k]) for k in ['LIMIT_BAL', 'AGE', 'EDUCATION'] + ST + BILLS + PAYS}
r = c.post('/score', json=payload).json(); print(r)
print('API score == batch score:', r['score'] == int(round(float(sc.score(test.iloc[[0]])[0]))))

{'status': 'ok', 'variables': ['LIMIT_BAL', 'PAY_1', 'PAY_3', 'PAY_5', 'PAY_6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT6', 'util', 'n_delq', 'bill_trend'], 'base_score': 600, 'pdo': 20}
{'score': 550, 'pd': 0.101, 'risk_band': 'Medium', 'reason_codes': [{'variable': 'PAY_AMT2', 'points_lost': 8.1}, {'variable': 'PAY_AMT3', 'points_lost': 5.2}, {'variable': 'PAY_AMT6', 'points_lost': 4.7}]}
API score == batch score: True
